In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/processed/heart_restored_missing.csv")
print(df.shape)
df.isna().sum()


(918, 13)


Age                 0
Sex                 0
ChestPainType       0
RestingBP          60
Cholesterol       201
FastingBS          90
RestingECG          2
MaxHR              55
ExerciseAngina     55
Oldpeak            62
ST_Slope          307
HeartDisease        0
source              0
dtype: int64

In [2]:
# Percentage of blanks per column, for each hospital
(df.isna().groupby(df["source"]).mean() * 100).round(1)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease,source
source,,,,,,,,,,,,,
cleveland,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
hungary,0.0,0.0,0.0,0.3,7.5,2.7,0.3,0.3,0.3,0.0,64.5,0.0,0.0
switzerland,0.0,0.0,0.0,1.6,100.0,61.0,0.8,0.8,0.8,4.9,13.8,0.0,0.0
va_longbeach,0.0,0.0,0.0,28.6,28.1,3.5,0.0,26.6,26.6,28.1,50.8,0.0,0.0


In [3]:
# Keep the label and hospital aside for evaluation later
labels = df[["HeartDisease", "source"]].copy()

# The features: everything else
features = df.drop(columns=["HeartDisease", "source"])

# Safeguard: stop with an error if a forbidden column slipped in
assert "HeartDisease" not in features.columns
assert "source" not in features.columns

print(features.shape)
print(features.columns.tolist())

(918, 11)
['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS', 'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope']


In [4]:
# The text (categorical) columns
cat_cols = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]

# Turn each category into its own 0/1 column
encoded = pd.get_dummies(features, columns=cat_cols, dtype=float)

# If the original value was blank, make its 0/1 columns blank too
for col in cat_cols:
    dummy_cols = [c for c in encoded.columns if c.startswith(col + "_")]
    encoded.loc[features[col].isna(), dummy_cols] = np.nan

print(encoded.shape)
print(encoded.columns.tolist())
encoded.isna().sum()

(918, 20)
['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak', 'Sex_F', 'Sex_M', 'ChestPainType_ASY', 'ChestPainType_ATA', 'ChestPainType_NAP', 'ChestPainType_TA', 'RestingECG_LVH', 'RestingECG_Normal', 'RestingECG_ST', 'ExerciseAngina_N', 'ExerciseAngina_Y', 'ST_Slope_Down', 'ST_Slope_Flat', 'ST_Slope_Up']


Age                    0
RestingBP             60
Cholesterol          201
FastingBS             90
MaxHR                 55
Oldpeak               62
Sex_F                  0
Sex_M                  0
ChestPainType_ASY      0
ChestPainType_ATA      0
ChestPainType_NAP      0
ChestPainType_TA       0
RestingECG_LVH         2
RestingECG_Normal      2
RestingECG_ST          2
ExerciseAngina_N      55
ExerciseAngina_Y      55
ST_Slope_Down        307
ST_Slope_Flat        307
ST_Slope_Up          307
dtype: int64

In [5]:
from sklearn.preprocessing import StandardScaler

num_cols = ["Age", "RestingBP", "Cholesterol", "MaxHR", "Oldpeak"]

scaler = StandardScaler()
scaled = encoded.copy()
scaled[num_cols] = scaler.fit_transform(encoded[num_cols])

# Check: averages near 0, standard deviations near 1
print(scaled[num_cols].describe().round(2))

# Check: the blanks are still there
print(scaled.isna().sum().sum(), "blanks in total")

          Age  RestingBP  Cholesterol   MaxHR  Oldpeak
count  918.00     858.00       717.00  863.00   856.00
mean    -0.00       0.00        -0.00   -0.00     0.00
std      1.00       1.00         1.00    1.00     1.00
min     -2.71      -2.82        -2.77   -2.99    -3.19
25%     -0.69      -0.66        -0.63   -0.68    -0.81
50%      0.05      -0.12        -0.12    0.09    -0.35
75%      0.69       0.42         0.51    0.75     0.57
max      2.49       3.66         6.09    2.49     4.88
1505 blanks in total


In [6]:
encoded.isna().sum().sum()

np.int64(1505)

In [7]:
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=5)
imputed_array = imputer.fit_transform(scaled)

# The result is a plain array; turn it back into a table with column names
imputed = pd.DataFrame(imputed_array, columns=scaled.columns, index=scaled.index)

print("Blanks left:", imputed.isna().sum().sum())
imputed[["Cholesterol", "ST_Slope_Down", "ST_Slope_Flat", "ST_Slope_Up"]].head(10)

Blanks left: 0


,Cholesterol,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up
0,0.719923,0.0,0.0,1.0
1,-1.142758,0.0,1.0,0.0
2,0.617390,0.2,0.4,0.4
3,-0.561738,0.0,1.0,0.0
4,0.723341,0.0,0.6,0.4
5,1.574363,0.0,0.2,0.8
6,-0.168695,0.0,0.4,0.6
7,-0.664271,0.2,0.2,0.6
8,-0.681360,0.0,1.0,0.0
9,0.634479,0.0,0.4,0.6


In [8]:
final = pd.DataFrame(index=imputed.index)

# 1. Undo scaling for the numeric columns
final[num_cols] = scaler.inverse_transform(imputed[num_cols])

# 2. FastingBS: majority vote -> 0 or 1
final["FastingBS"] = (imputed["FastingBS"] >= 0.5).astype(int)

# 3. Categories: the column with the most votes wins
for col in cat_cols:
    dummy_cols = [c for c in imputed.columns if c.startswith(col + "_")]
    winner = imputed[dummy_cols].idxmax(axis=1)
    final[col] = winner.str.replace(col + "_", "")

# Count ties in ST_Slope
slope_cols = ["ST_Slope_Down", "ST_Slope_Flat", "ST_Slope_Up"]
top = imputed[slope_cols].max(axis=1)
n_ties = (imputed[slope_cols].eq(top, axis=0).sum(axis=1) > 1).sum()
print("ST_Slope ties:", n_ties)

final.head(10)

ST_Slope ties: 22


,Age,RestingBP,Cholesterol,MaxHR,Oldpeak,FastingBS,Sex,ChestPainType,RestingECG,ExerciseAngina,ST_Slope
0,40.0,140.0,289.0,172.0,0.0,0,M,ATA,Normal,N,Up
1,49.0,160.0,180.0,156.0,1.0,0,F,NAP,Normal,N,Flat
2,37.0,130.0,283.0,98.0,0.0,0,M,ATA,ST,N,Flat
3,48.0,138.0,214.0,108.0,1.5,0,F,ASY,Normal,Y,Flat
4,54.0,150.0,289.2,122.0,0.0,0,M,NAP,Normal,N,Flat
5,39.0,120.0,339.0,170.0,0.0,0,M,NAP,Normal,N,Up
6,45.0,130.0,237.0,170.0,0.0,0,F,ATA,Normal,N,Up
7,54.0,110.0,208.0,142.0,0.0,0,M,ATA,Normal,N,Up
8,37.0,140.0,207.0,130.0,1.5,0,M,ASY,Normal,Y,Flat
9,48.0,120.0,284.0,120.0,0.0,0,F,ATA,Normal,N,Up


In [9]:
# The original features, before imputation
original = df.drop(columns=["HeartDisease", "source"])

for col in original.columns:
    known = original[col].notna()   # cells that were NOT blank
    if col in num_cols:
        # Numbers: allow tiny rounding differences from scaling back
        unchanged = np.isclose(final.loc[known, col], original.loc[known, col]).all()
    else:
        unchanged = (final.loc[known, col] == original.loc[known, col]).all()
    print(f"{col}: real values unchanged = {unchanged}")

Age: real values unchanged = True
Sex: real values unchanged = True
ChestPainType: real values unchanged = True
RestingBP: real values unchanged = True
Cholesterol: real values unchanged = True
FastingBS: real values unchanged = True
RestingECG: real values unchanged = True
MaxHR: real values unchanged = True
ExerciseAngina: real values unchanged = True
Oldpeak: real values unchanged = True
ST_Slope: real values unchanged = True


In [10]:
was_blank = df["Cholesterol"].isna()

compare = pd.DataFrame({
    "real": final.loc[~was_blank, "Cholesterol"].describe(),
    "imputed": final.loc[was_blank, "Cholesterol"].describe(),
}).round(1)
print(compare)

# Filled-in cholesterol, broken down by hospital
print(final.loc[was_blank, "Cholesterol"].groupby(df.loc[was_blank, "source"]).describe().round(1))

        real  imputed
count  717.0    201.0
mean   246.9    245.5
std     58.6     23.8
min     85.0    185.0
25%    210.0    230.6
50%    240.0    246.4
75%    277.0    260.8
max    603.0    339.0
              count   mean   std    min    25%    50%    75%    max
source                                                             
hungary        22.0  239.5  30.3  197.6  214.4  233.3  265.6  300.0
switzerland   123.0  246.4  23.6  185.0  231.6  247.8  260.0  339.0
va_longbeach   56.0  246.0  21.5  204.0  231.2  244.3  261.2  291.0


In [11]:
# Put the columns back in the original order, then add the label and hospital
heart_imputed = final[original.columns].copy()
heart_imputed["HeartDisease"] = labels["HeartDisease"]
heart_imputed["source"] = labels["source"]

# Final checks
print(heart_imputed.shape)
print("Blanks:", heart_imputed.isna().sum().sum())

heart_imputed.to_csv("data/processed/heart_imputed.csv", index=False)
print("Saved.")

(918, 13)
Blanks: 0
Saved.


## KNN imputation

**Input:** `data/processed/heart_restored_missing.csv` (918 patients, original UCI blanks restored).

**Leakage safeguard:** `HeartDisease` and `source` were set aside before imputation and checked with `assert`. Imputation used the 11 features only.

### Method
1. **One-hot encoding** of the categorical features (Sex, ChestPainType, RestingECG, ExerciseAngina, ST_Slope) → 20 columns. Blank categories kept as blank in all their 0/1 columns.
2. **Standardisation** (mean 0, SD 1) of the continuous features: Age, RestingBP, Cholesterol, MaxHR, Oldpeak. FastingBS and the one-hot columns left as 0/1.
3. **KNN imputation** with k = 5 (scikit-learn `KNNImputer`, distances computed on available values only).
4. **Back-transformation:**
   - Continuous features: scaling reversed to original units (e.g. Cholesterol in mg/dL).
   - FastingBS: neighbour average ≥ 0.5 → 1, otherwise 0.
   - Categorical features: majority vote among the 5 neighbours (highest one-hot average).
   - ST_Slope ties: 22 of 307 imputed values (7%), resolved by column order.

### Validation
- Blanks before: 1,505 cells (after encoding). Blanks after: 0.
- Real (non-blank) values unchanged in all 11 features.
- Imputed cholesterol (n = 201) vs real (n = 717):
  - Mean: 245.5 vs 246.9 mg/dL (similar)
  - SD: 23.8 vs 58.6 (much smaller)
  - Range: 185–339 vs 85–603 mg/dL
- By hospital: Switzerland n = 123 (all of its patients), VA n = 56, Hungary n = 22.

### Limitations
- **Variance shrinkage:** averaging 5 neighbours pulls imputed values towards the middle, so imputed cholesterol is much less spread out than real cholesterol.
- **Switzerland:** 100% of cholesterol values imputed from patients at other hospitals. This assumes Swiss patients resemble similar patients elsewhere, which cannot be checked. Swiss patients may look artificially alike on cholesterol, which could push clusters towards recovering the hospital.
- These are addressed by the planned sensitivity analyses: (1) excluding Switzerland, (2) excluding Cholesterol, (3) using Kaggle's own filled values.

**Output:** `data/processed/heart_imputed.csv` (918 × 13, no blanks). This is the input for the main analysis.